# Zero-shot TAD / DADA-2000 — **SA² (R4) vs its host (W0)**, Nexar-trained, seed 2024

Rule (written before any number): `core/docs/final/ZS_TAD_DADA.md` §3. Prerequisite: `zs_pack_frames.ipynb` (CPU)
has archived the row frames of both sets.

| arm | what | prefix | checkpoint |
|---|---|---|---|
| **W0** (control) | CLIP + V2-S via `W_u` (zero-init) — **= v2 Nexar A2 s2024** (DoTA-CAP-dev 0.8110072 in both) | OFF | `outputs/wu/W0/s2024` (trained here only if not verified) |
| **R4** | W0 + SA² on the top-12 motion-selected CLIP patches | OFF | `outputs/stp/R4/s2024` |

| step | what | GPU? |
|---|---|---|
| 1–3 | setup, deps, helpers, ids, preflight (archives, bakes, checkpoints, configs) | no |
| 4 | W0 s2024: reuse if verified, else train it (v2 A2's recipe, `wu_arms.ipynb`'s command) | yes, only if missing |
| 5 | patches: `clip_windows packed` (1×1 + 7×7 of the 224² squash; 1×1 gated vs the CLIP rows) → `stp_patches select` (top-12, raw Mo) | **yes** |
| 6 | **TAD**: `core.evaluate` × 2 arms (§26: s8, raw, `--save-scores`) | yes |
| 7 | **DADA**: `dada_whole_eval` × 2 arms (370 sources, whole video, s8) | yes |
| 8 | `zs_readout` → **paste `REPORTS/zs_readout.md` to Claude** | no |

Inputs = v2's A2 bakes with the **Nexar** statistics. Every step skips what is already on Drive.
One seed ⇒ every sentence is **PRELIMINARY (n = 1 seed)** and adopts nothing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (TAD, DADA2000_orig, DADA2000Origin), cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes (Nexar stats), KNN, v2 A2 reads
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every archive / cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'     # new caches (clip_windows/, clip_patches/) land here
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'ZS_ARMS'), 'this upload predates the zero-shot TAD / DADA build -- re-upload kat-vad/'
assert (REPO / 'core/docs/final/ZS_TAD_DADA.md').is_file(), 'no ZS_TAD_DADA.md (the read-out rule) in this upload'

STRIDE   = constants.ZS_STRIDE                                            # 8
TAD      = constants.DATA_ROOT / constants.TAD_DATASET                    # Thesis/data/TAD (labels s8)
TAD_NCC  = Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc'                    # T0's CLIP rows = frames[::8]
T2NAME   = constants.DADA_ORIGIN_DATASET                                  # DADA2000_orig
T2       = constants.DATA_ROOT / T2NAME                                   # meta.json (T2-test windows)
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME                       # per-source CLIP rows at s8
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ORIG     = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
ARCHIVE  = Path(DRIVE_F) / 'data' / constants.ZS_FRAMES_DIRNAME            # written by zs_pack_frames.ipynb
ARC      = {'tad': ARCHIVE / constants.TAD_DATASET, 'dada': ARCHIVE / T2NAME}
CLIP     = {'tad': TAD_NCC, 'dada': T2_CLIP}
ROOT     = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME            # Thesis-Final/outputs/zs
REPORTS  = ROOT / 'REPORTS'

SEED     = constants.ZS_SEED                                              # 2024
ARMS     = list(constants.ZS_ARMS)                                        # ['W0', 'R4']
CONTROL, SA2 = constants.ZS_CONTROL_ARM, constants.ZS_SA2_ARM
ENC      = constants.V5_ENCODER                                           # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW
K, MOTION, CELLS = constants.STP_TOP_K, constants.STP_MOTION_RAW, constants.STP_CELLS_RESIZED

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
V2_ROOT  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR                          # v2's eval/{tad,dada}/A2 (printed)
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{C}'      # Nexar_window: training + defs.json
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                         # W0's training rows (+ the stats)
KNN      = V2_CACHE / 'knn' / f'{constants.NEXAR_DATASET}_{C}' / constants.KNN_CACHE_FILENAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'
BAKE     = {'tad': V2_CACHE / 'v2' / 'A2_S' / f'{constants.TAD_DATASET}_s8_nexar',
            'dada': V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'}
ANNOT    = ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME
RUN_DIR  = {CONTROL: constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / CONTROL / f's{SEED}',
            SA2: constants.OUTPUT_ROOT / constants.STP_OUTPUT_DIRNAME / SA2 / f's{SEED}'}

from core.data import stp_patches as stp_data      # noqa: E402
from core.tools import clip_windows               # noqa: E402

DSET     = {'tad': constants.TAD_DATASET, 'dada': T2NAME}
WIN      = {s: clip_windows.cache_dir(DSET[s], STRIDE, constants.STP_WINDOW_GRIDS, CELLS) for s in DSET}
PATCH    = {s: stp_data.cache_dir(DSET[s], STRIDE, MOTION, K) for s in DSET}
LOCAL    = Path('/content/zs')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('TAD_ncc', TAD_NCC),
                   ('T2 meta', T2 / constants.META_FILENAME), ('T2 CLIP', T2_CLIP), ('T2 census', T2_CENSUS),
                   ('DADA annotation', ANNOT), ('Nexar_window', DATA), ('TAD A2 bake', BAKE['tad']),
                   ('DADA A2 bake', BAKE['dada']), ('Nexar A2 bake', A2_NEXAR), ('v2 A2 TAD read', V2_ROOT / 'eval' / 'tad' / 'A2'),
                   ('v2 A2 DADA read', V2_ROOT / 'eval' / 'dada' / 'A2'), ('R4 run', RUN_DIR[SA2])):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:16s} {path}')
print('windows ->', WIN, '\npatches ->', PATCH, '\nreads   ->', ROOT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 4-7 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None, quiet_steps=False, stdout_to=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    out = open(stdout_to, 'w') if stdout_to else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                if out:
                    out.write(line)
                    continue
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'epoch' in line and 'batch' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
        if out:
            out.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])

import torch
import yaml


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + manifests if given) to VM disk once (bit-identical; C2)."""
    dst.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi))."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


from core.data.v2_splits import lines_sha1             # noqa: E402
from core.tools.dada_whole_eval import test_sources    # noqa: E402

labels_s8 = json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text())
IDS = {'tad': sorted(labels_s8), 'dada': test_sources(T2)}
n_abn = sum(int(max(g) > 0) for g in labels_s8.values())
assert (len(IDS['tad']), n_abn) == (constants.V2_TADZ_VIDEOS, constants.V2_TADZ_ABNORMAL), (len(IDS['tad']), n_abn)
assert len(IDS['dada']) == 370, len(IDS['dada'])
for s in ('tad', 'dada'):
    miss = [v for v in IDS[s] if not (CLIP[s] / f'{v}.npy').exists()]
    assert not miss, f'{s}: {len(miss)} videos have no CLIP rows in {CLIP[s]}: {miss[:5]}'
bad = [v for v in IDS['tad'] if rows(TAD_NCC, v) != len(labels_s8[v])]
assert not bad, f'TAD_ncc rows != s8 labels on {bad[:5]}'
ID_FILE = {s: write_ids(LOCAL / f'{s}_ids.txt', IDS[s]) for s in IDS}


def archive_ok(s):
    """The verify manifest exists and covers exactly this id list (zs_frames verify, PASS only)."""
    m = ARC[s] / constants.ZS_FRAMES_MANIFEST
    if not m.is_file():
        return False
    man = json.loads(m.read_text())
    return man['pass'] and man['ids_sha1'] == lines_sha1(IDS[s]) and man['stride'] == STRIDE


print(f"TAD {len(IDS['tad'])} test videos ({n_abn} abnormal, {sum(rows(TAD_NCC, v) for v in IDS['tad'])} rows) | "
      f"DADA {len(IDS['dada'])} T2-test sources ({sum(rows(T2_CLIP, v) for v in IDS['dada'])} rows)")
print({s: ('archived' if archive_ok(s) else 'NOT archived') for s in IDS})

assert archive_ok('tad') and archive_ok('dada'), 'frame archives missing -- run colab/final/zs_pack_frames.ipynb first'
for s in ('tad', 'dada'):                         # inputs: v2 A2 bakes with the Nexar statistics, rows == CLIP rows
    man = json.loads((BAKE[s] / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (man['crn'], man['motion']) == (constants.V2_OFF, ENC), f'{s} bake is {man}'
    bad = [v for v in IDS[s] if rows(BAKE[s], v) != rows(CLIP[s], v)]
    assert not bad, f'{s}: bake rows != CLIP rows on {bad[:5]}'
nexar_sha = json.loads((A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1']
for s in ('tad', 'dada'):
    sha = json.loads((BAKE[s] / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1']
    assert sha == nexar_sha, f'{s} bake carries statistics {sha[:12]}, the training bake {nexar_sha[:12]}'
print(f'bakes OK: A2 (no CRN, V2-S), Nexar statistics {nexar_sha[:12]}, rows == CLIP rows on both sets')

## 4. W0 s2024 — reuse, or train it (only if Drive has no verified copy)

`wu_arms.ipynb` trained W0 with the code of this branch; its DoTA-CAP-dev read equals v2 Nexar A2 s2024 to the last
digit. If `verified.json` is missing, the same command trains it again here (v2 A2's recipe, `stp.*` off, `W_u`
zero-init, LR ×1). Then both checkpoints are checked: J10, the arm's config, **prefix OFF** on both.

In [ ]:
from core.tools.v5_stage1 import _flat    # noqa: E402


def stage_tree(src, dst):
    """A whole Drive dir (Nexar_window) to VM disk once."""
    if not dst.exists():
        tmp = dst.with_name(dst.name + '.partial')
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        os.replace(tmp, dst)
    return dst


def train_w0():
    drive_dir, local = RUN_DIR[CONTROL], LOCAL / 'runs' / CONTROL / f's{SEED}'
    if is_verified(drive_dir):
        print(f'{CONTROL} s{SEED}: verified on Drive -- not retrained')
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    data = stage_tree(DATA, LOCAL / 'stage' / f'data_{C}')
    a2 = stage_tree(A2_NEXAR, LOCAL / 'stage' / 'A2_s8')
    knn = LOCAL / 'stage' / 'knn.npz'
    if not knn.exists():
        shutil.copy2(KNN, knn)
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={constants.NEXAR_DATASET}',
           '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}',
           *[x for k, v in constants.WU_ARMS[CONTROL].items() for x in ('--set', f'{k}={v}')],
           '--data-dir', data, '--clip-dir', a2, '--knn-cache', knn, '--output-dir', local, *resume]
    t0 = time.time()
    run(cmd, log=LOCAL / f'train_{CONTROL}.log', quiet_steps=True)
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, 'J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{CONTROL} s{SEED}: trained in {(time.time() - t0) / 60:.0f} min')


train_w0()

# J10 + config: both arms are the Nexar host at s2024 with the released prompts (prefix OFF)
ref = _flat(yaml.safe_load((A2_RUNS / f's{SEED}' / 'config.yaml').read_text()))
WANT_STP = {CONTROL: {'stp.sa2': False}, SA2: {'stp.sa2': True, 'stp.motion': MOTION, 'stp.top_k': K}}
for arm in ARMS:
    d = RUN_DIR[arm]
    step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
    assert last is not None and step == last['global_step'] == STEPS, f'{arm}: J10 (ckpt {step}, want {STEPS})'
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    assert got['stp.text_prompt'] == constants.STP_PROMPT_RELEASED, f'{arm}: prefix is not OFF ({got["stp.text_prompt"]})'
    assert all(got[k] == v for k, v in WANT_STP[arm].items()), (arm, {k: got[k] for k in WANT_STP[arm]})
    assert (got['v2.crn'], got['v2.motion'], got['train.seed']) == (constants.V2_OFF, ENC, SEED), arm
    knobs = (got.get('model.motion_init_std', 0.0), got.get('train.motion_lr_mult', 1.0))
    assert knobs == (0.0, 1.0), f'{arm}: W_u is not the shipped host (init std, LR mult) = {knobs}'
    diff = {k: (ref.get(k), got.get(k)) for k in set(ref) | set(got)
            if not k.startswith(('stp.', 'v5.')) and k not in ('model.motion_init_std', 'train.motion_lr_mult')
            and ref.get(k) != got.get(k) and not (k == 'model.motion_dropout' and not got.get(k))}
    assert not diff, f'{arm} differs from v2 A2 s{SEED} beyond stp / v5 / W_u knobs: {diff}'
    print(f'{arm} s{SEED}: J10 {STEPS}, config == v2 A2 s{SEED} + stp {{{", ".join(f"{k}={got[k]}" for k in WANT_STP[arm])}}}, prefix OFF')

## 5. Patch caches (GPU): windows from the archives → top-12 by Mo

`clip_windows packed` unpacks each `{id}.zip` to VM disk, encodes 1×1 + 7×7 cells of the 224² squash, and gates the
1×1 cell against the video's own CLIP rows (row count, every row cos ≥ 0.95, mean ≥ 0.99 — same pixels): a failing
video stores nothing. `stp_patches select` keeps the top-12 by raw Mo (±1 row). Then C13: patch rows == bake rows.
**Re-run after a disconnect**; finished videos are skipped.

In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU'
for s in ('tad', 'dada'):
    t0 = time.time()
    todo = [v for v in IDS[s] if not (WIN[s] / f'{v}.npy').exists()]
    print(f'--- {s}: windows {len(IDS[s]) - len(todo)}/{len(IDS[s])} done')
    if todo:
        arc = stage(ARC[s], LOCAL / 'arc' / s)                 # ≈ 0.3-2.3 GB of zips, once
        clip = stage(CLIP[s], LOCAL / 'clip' / s, IDS[s])
        run([sys.executable, '-m', 'core.tools.clip_windows', 'packed', '--archive-dir', arc,
             '--work-dir', LOCAL / 'unzip', '--clip-dir', clip, '--ids-file', ID_FILE[s],
             '--report', REPORTS / f'zs_windows_{s}.json', '--output-dir', WIN[s],
             '--grids', 1, constants.STP_GRID, '--cells', CELLS, '--device', 'auto'], log=LOCAL / f'windows_{s}.log')
        print(f'{s} windows: {(time.time() - t0) / 60:.1f} min')
    report = json.loads((REPORTS / f'zs_windows_{s}.json').read_text())
    missing = {v: report.get(v, {}).get('reason', 'not_run') for v in IDS[s] if not (WIN[s] / f'{v}.npy').exists()}
    assert not missing, f'{s}: {len(missing)} videos have no window cache: {dict(list(missing.items())[:10])}'
    corpus = clip_windows.corpus_gate([report[v] for v in IDS[s] if v in report])
    print(f'{s} 1x1 gate:', corpus)
    assert corpus['reason'] == 'ok', f'{s}: window transform disagrees with the CLIP cache: {corpus}'
    run([sys.executable, '-m', 'core.tools.stp_patches', 'select', '--windows-dir', WIN[s], '--ids-file', ID_FILE[s],
         '--cells', CELLS, '--motion', MOTION, '--top-k', K, '--out-dir', PATCH[s],
         '--report', REPORTS / f'zs_select_{s}.json'])
    bad = [v for v in IDS[s] if np.load(PATCH[s] / f'{v}.npy', mmap_mode='r').shape != (rows(BAKE[s], v), K * constants.STP_PATCH_DIM)]
    assert not bad, f'{s}: patch rows != bake rows on {bad[:5]} (C13)'
    print(f'{s} patch cache OK: {len(IDS[s])} videos, rows == A2 bake rows')
flush_drive()

## 6. TAD — §26's protocol per arm (`core.evaluate`, s8, raw scores, `--save-scores`)

Exactly v2's TAD command (`nexar_zeroshot.ipynb` step 4) on the A2 bake; R4 adds `--patch-dir`. W0 is printed beside
v2 A2 s2024 (0.6983 macro): the same function at the same seed should reproduce it.

In [ ]:
L_TAD = LOCAL / 'TAD'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_BAKE = {s: stage(BAKE[s], LOCAL / 'bake' / s, IDS[s]) for s in ('tad', 'dada')}
L_PATCH = {s: stage(PATCH[s], LOCAL / 'patch' / s, IDS[s]) for s in ('tad', 'dada')}
wrote = False
for arm in ARMS:
    out = ROOT / 'eval' / 'tad' / arm / f's{SEED}'
    if (out / 'results.json').exists():
        print(f'tad {arm}: already read')
        continue
    local = LOCAL / 'tad' / arm / f's{SEED}'
    shutil.rmtree(local, ignore_errors=True)
    patch = ['--patch-dir', L_PATCH['tad']] if arm == SA2 else []
    run([sys.executable, '-m', 'core.evaluate', '--ckpt', RUN_DIR[arm] / 'checkpoint_last.pt',
         '--set', f'data.dataset={constants.TAD_DATASET}', '--data-dir', L_TAD, '--clip-dir', L_BAKE['tad'],
         *patch, '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'tad_{arm}.log')
    publish(local, out)
    wrote = True
for arm in ARMS:
    r = json.loads((ROOT / 'eval' / 'tad' / arm / f's{SEED}' / 'results.json').read_text())
    print(f'tad {arm} s{SEED}: macro {r["auc_macro"]:.4f} over {r["auc_macro_videos"]} | micro {r["auc"]:.4f} | norm {r["score_norm"]}')
v2 = json.loads((V2_ROOT / 'eval' / 'tad' / 'A2' / f's{SEED}' / 'results.json').read_text())
print(f'printed: v2 A2 s{SEED} macro {v2["auc_macro"]:.4f} | micro {v2["auc"]:.4f}  (W0 should reproduce it)')
if wrote:
    flush_drive()

## 7. DADA-2000 whole videos — 370 T2-test sources, every s8 row (`dada_whole_eval`)

v2's DADA command (`nexar_zeroshot.ipynb` step 5) with one checkpoint per arm; R4 adds `--patch-dir`.

In [ ]:
L_T2_META = LOCAL / 'T2_meta'
L_T2_META.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2_META / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):               # defs.json etc. -- the class definitions the arms trained with
    shutil.copy2(f, L_DEFS / f.name)
wrote = False
for arm in ARMS:
    out = ROOT / 'eval' / 'dada' / arm
    if (out / 'dada_readout.json').exists():
        print(f'dada {arm}: already read')
        continue
    local = LOCAL / 'dada' / arm
    patch = ['--patch-dir', L_PATCH['dada']] if arm == SA2 else []
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', RUN_DIR[arm] / 'checkpoint_last.pt',
         '--t2-dir', L_T2_META, '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', L_BAKE['dada'],
         *patch, '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'dada_{arm}.log')
    publish(local, out)
    wrote = True
for arm in ARMS:
    r = json.loads((ROOT / 'eval' / 'dada' / arm / 'dada_readout.json').read_text())
    print(f"dada {arm} s{SEED}: macro {r['macro']['mean']:.4f} over {r['two_class']} sources | micro {r['micro']:.4f}")
v2 = json.loads((V2_ROOT / 'eval' / 'dada' / 'A2' / 'dada_readout.json').read_text())
print(f"printed: v2 A2 s{SEED} macro {v2['per_run_macro'][f's{SEED}']:.4f} (W0 should reproduce it); "
      f"5-seed mean {v2['macro']['mean']:.4f}")
if wrote:
    flush_drive()

## 8. Read-out → **paste `REPORTS/zs_readout.md` to Claude**

Rule `ZS_TAD_DADA.md` §3, CPU only.

In [ ]:
run([sys.executable, '-m', 'core.tools.zs_readout', '--root', ROOT, '--v2-root', V2_ROOT,
     '--runs', CONTROL, RUN_DIR[CONTROL], '--runs', SA2, RUN_DIR[SA2], '--seed', SEED])
print((REPORTS / 'zs_readout.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/zs_readout.md to Claude')